# Runoff Forecasting Backend Integration

This notebook demonstrates how to use the FastAPI backend to:
- Connect to the trained ML models
- Make predictions on runoff data
- Compare model performance
- Visualize results

**Requirements:** Ensure the backend is running with `python startup.py`

## 1) Set Up Environment and Imports

Import core libraries, configure plotting, and set a random seed for reproducibility.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

SEED = 42
np.random.seed(SEED)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

print("Libraries imported successfully")
print(f"Random seed set to {SEED}")

## 2) Load and Inspect Dataset

Load a CSV dataset. If no dataset exists yet, this notebook creates a realistic synthetic runoff dataset and saves it as `data/runoff_dataset.csv` so the workflow remains fully runnable.

In [ ]:
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

csv_path = os.path.join(data_dir, "runoff_dataset.csv")

if not os.path.exists(csv_path):
    n = 1200
    rainfall = np.random.gamma(shape=2.5, scale=8.0, size=n)
    temperature = np.random.normal(loc=24, scale=6, size=n)
    humidity = np.random.uniform(35, 95, size=n)
    soil_moisture = np.random.uniform(0.1, 0.9, size=n)
    evapotranspiration = np.random.uniform(1.0, 8.0, size=n)
    land_use = np.random.choice(["urban", "agriculture", "forest"], size=n, p=[0.25, 0.45, 0.30])
    soil_type = np.random.choice(["sandy", "loamy", "clayey"], size=n, p=[0.3, 0.45, 0.25])

    land_use_effect = pd.Series(land_use).map({"urban": 8.5, "agriculture": 5.0, "forest": 2.5}).values
    soil_effect = pd.Series(soil_type).map({"sandy": -2.0, "loamy": 1.5, "clayey": 4.0}).values

    noise = np.random.normal(0, 3.5, size=n)
    runoff = (
        0.75 * rainfall
        + 10.0 * soil_moisture
        - 0.55 * evapotranspiration
        + 0.08 * humidity
        - 0.12 * temperature
        + land_use_effect
        + soil_effect
        + noise
    )
    runoff = np.clip(runoff, a_min=0, a_max=None)

    synthetic_df = pd.DataFrame(
        {
            "rainfall": rainfall,
            "temperature": temperature,
            "humidity": humidity,
            "soil_moisture": soil_moisture,
            "evapotranspiration": evapotranspiration,
            "land_use": land_use,
            "soil_type": soil_type,
            "runoff": runoff,
        }
    )

    for col in ["rainfall", "humidity", "soil_type"]:
        idx = np.random.choice(n, size=int(0.02 * n), replace=False)
        synthetic_df.loc[idx, col] = np.nan

    synthetic_df.to_csv(csv_path, index=False)
    print(f"Synthetic dataset created at: {csv_path}")

# Load dataset
df = pd.read_csv(csv_path)
print(f"Dataset loaded from: {csv_path}")
print(f"Shape: {df.shape}")

display(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nNumeric summary:")
display(df.describe(include=[np.number]).T)

## 3) Preprocess Features and Target

Define feature matrix and target vector. Build preprocessing pipelines for numeric and categorical columns.

In [ ]:
target_col = "runoff"
X = df.drop(columns=[target_col])
y = df[target_col]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)
print(f"X shape: {X.shape}, y shape: {y.shape}")

## 4) Split Data for Training and Testing

Create train and test sets and verify target distribution consistency.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("Train/Test shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}, y_test : {y_test.shape}")

print("\nTarget distribution check:")
print(f"y_train mean={y_train.mean():.3f}, std={y_train.std():.3f}")
print(f"y_test  mean={y_test.mean():.3f}, std={y_test.std():.3f}")

## 5) Train a Baseline Model

Build an end-to-end pipeline (preprocessing + regressor) and train it.

In [ ]:
baseline_pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_pipeline.fit(X_train, y_train)
print("Baseline model trained successfully")

## 6) Evaluate Model Performance

Compute MAE, RMSE, and $R^2$. Then plot predicted vs actual and residuals.

In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

baseline_preds = baseline_pipeline.predict(X_test)
baseline_mae, baseline_rmse, baseline_r2 = regression_metrics(y_test, baseline_preds)

print("Baseline Performance:")
print(f"MAE : {baseline_mae:.4f}")
print(f"RMSE: {baseline_rmse:.4f}")
print(f"R²  : {baseline_r2:.4f}")

residuals = y_test - baseline_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, baseline_preds, alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_title("Predicted vs Actual")
axes[0].set_xlabel("Actual Runoff")
axes[0].set_ylabel("Predicted Runoff")

sns.histplot(residuals, kde=True, ax=axes[1], color="purple")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual (Actual - Predicted)")

plt.tight_layout()
plt.show()

## 7) Tune Hyperparameters with Cross-Validation

Run `GridSearchCV` on the model pipeline and evaluate the tuned model on the test set.

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
}

grid = GridSearchCV(
    estimator=Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", RandomForestRegressor(random_state=SEED, n_jobs=-1)),
        ]
    ),
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
    verbose=1,
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)
print(f"Best CV RMSE: {-grid.best_score_:.4f}")

tuned_model = grid.best_estimator_
tuned_preds = tuned_model.predict(X_test)
tuned_mae, tuned_rmse, tuned_r2 = regression_metrics(y_test, tuned_preds)

print("\nTuned Model Test Performance:")
print(f"MAE : {tuned_mae:.4f}")
print(f"RMSE: {tuned_rmse:.4f}")
print(f"R²  : {tuned_r2:.4f}")

print("\nImprovement over baseline (negative means better for error metrics):")
print(f"ΔMAE : {tuned_mae - baseline_mae:+.4f}")
print(f"ΔRMSE: {tuned_rmse - baseline_rmse:+.4f}")
print(f"ΔR²  : {tuned_r2 - baseline_r2:+.4f}")

## 8) Persist Model and Run Inference

Save the tuned pipeline with `joblib`, reload it, and run sample inference.

In [ ]:
artifacts_dir = "artifacts"
os.makedirs(artifacts_dir, exist_ok=True)

model_path = os.path.join(artifacts_dir, "runoff_regressor_pipeline.joblib")
joblib.dump(tuned_model, model_path)
print(f"Model saved to: {model_path}")

loaded_model = joblib.load(model_path)
print("Model reloaded successfully")

new_samples = X_test.sample(5, random_state=SEED).copy()
inference_preds = loaded_model.predict(new_samples)

inference_df = new_samples.copy()
inference_df["predicted_runoff"] = inference_preds

print("\nInference preview:")
display(inference_df.head())